In [1]:
import json
from urllib.request import Request, urlopen
TEAM_ID = "TEAM_09"
API_KEY = "oc_AWwmZ4Xep7YD7U0QUy5kqCt1lZLqcR-L"
API_URL = "https://bftrxasgtunepckchcoz.supabase.co/functions/v1/oracle"

def api(payload):
    req = Request(API_URL, data=json.dumps(payload).encode(),
        headers={"Content-Type":"application/json", "X-API-Key":API_KEY})
    with urlopen(req, timeout=30) as r:
        return json.load(r)
    
spec = api({"action":"spec", "team_id":TEAM_ID})
SPACE = spec["hyperparameters"]
print("Model:", spec["model"])
for name, values in SPACE.items():
    print(name, ":", values)

Model: DecisionTreeRegressor
splitter : ['best', 'random']
criterion : ['squared_error', 'friedman_mse']
max_depth : [2, 3, 4, 5, 6, 8, 10, None]
max_features : [0.4, 0.6, 0.8, 1]
min_samples_leaf : [1, 2, 4, 8]
min_samples_split : [2, 5, 10, 20]


In [48]:
results = []

def oracle_query(params):
    result = api({"action":"query", "team_id":TEAM_ID, "params":params})
    loss =  float(result["loss"])

    results.append({
        "params": params.copy(),
        "loss": loss
    })

    print("Params:", params)
    print("Loss:", loss)

    return loss


In [49]:
best_params = {
    "criterion": "squared_error",
    "splitter": "best",
    "max_depth": 3,
    "min_samples_split": 10,
    "min_samples_leaf": 2,
    "max_features": 1
}

best_loss = oracle_query(best_params)

print("Initial loss:", best_loss)

Params: {'criterion': 'squared_error', 'splitter': 'best', 'max_depth': 3, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': 1}
Loss: 26.2917253751272
Initial loss: 26.2917253751272


In [50]:
max_depth_values = [2, 3, 4, 5, 6, 8, 10, None]

for depth in max_depth_values:

    params = best_params.copy()
    params["max_depth"] = depth

    loss = oracle_query(params)

    if loss < best_loss:
        best_loss = loss
        best_params = params.copy()

print("\nBest so far:")
print(best_params)
print("Loss:", best_loss)

Params: {'criterion': 'squared_error', 'splitter': 'best', 'max_depth': 2, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': 1}
Loss: 28.4768017970479
Params: {'criterion': 'squared_error', 'splitter': 'best', 'max_depth': 3, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': 1}
Loss: 26.2917253751272
Params: {'criterion': 'squared_error', 'splitter': 'best', 'max_depth': 4, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': 1}
Loss: 28.1201173221294
Params: {'criterion': 'squared_error', 'splitter': 'best', 'max_depth': 5, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': 1}
Loss: 31.7432933246309
Params: {'criterion': 'squared_error', 'splitter': 'best', 'max_depth': 6, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': 1}
Loss: 34.5656874771315
Params: {'criterion': 'squared_error', 'splitter': 'best', 'max_depth': 8, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': 1}
Loss: 34.9444908922272
Params: {'

#### max_depth=3 gives the lowest loss, investigating min_samples_leaf

In [51]:
min_leaf_values = [1, 2, 4, 8]

for leaf in min_leaf_values:

    params = best_params.copy()
    params["min_samples_leaf"] = leaf

    loss = oracle_query(params)

    if loss < best_loss:
        best_loss = loss
        best_params = params.copy()

print("\nBest so far:")
print(best_params)
print("Loss:", best_loss)

Params: {'criterion': 'squared_error', 'splitter': 'best', 'max_depth': 3, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 1}
Loss: 26.2917253751272
Params: {'criterion': 'squared_error', 'splitter': 'best', 'max_depth': 3, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': 1}
Loss: 26.2917253751272
Params: {'criterion': 'squared_error', 'splitter': 'best', 'max_depth': 3, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_features': 1}
Loss: 26.2917253751272
Params: {'criterion': 'squared_error', 'splitter': 'best', 'max_depth': 3, 'min_samples_split': 10, 'min_samples_leaf': 8, 'max_features': 1}
Loss: 26.2917253751272

Best so far:
{'criterion': 'squared_error', 'splitter': 'best', 'max_depth': 3, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': 1}
Loss: 26.2917253751272


#### max_depth=3 and min_samples_leaf=2, exploring max_features

In [52]:
max_features_values = [0.4, 0.6, 0.8, 1]

for feat in max_features_values:

    params = best_params.copy()
    params["max_features"] = feat

    loss = oracle_query(params)

    if loss < best_loss:
        best_loss = loss
        best_params = params.copy()

print("\nBest so far:")
print(best_params)
print("Loss:", best_loss)

Params: {'criterion': 'squared_error', 'splitter': 'best', 'max_depth': 3, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': 0.4}
Loss: 40.5390993002183
Params: {'criterion': 'squared_error', 'splitter': 'best', 'max_depth': 3, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': 0.6}
Loss: 33.1423044662884
Params: {'criterion': 'squared_error', 'splitter': 'best', 'max_depth': 3, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': 0.8}
Loss: 28.3058470255367
Params: {'criterion': 'squared_error', 'splitter': 'best', 'max_depth': 3, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': 1}
Loss: 26.2917253751272

Best so far:
{'criterion': 'squared_error', 'splitter': 'best', 'max_depth': 3, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': 1}
Loss: 26.2917253751272


In [53]:
print("\nBest configuration:")
print(best_params)

print("Best loss:", best_loss)

print("Total API calls:", len(results))


Best configuration:
{'criterion': 'squared_error', 'splitter': 'best', 'max_depth': 3, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': 1}
Best loss: 26.2917253751272
Total API calls: 17
